# 🧠 ContentFlow Agents — Final Multi-Agent Content Writer

A production-minded LangGraph design with:

- GPT-5 routing and content agents
- A Deep Research sub-agent powered by Tavily
- Advanced Tavily search for trends, keywords, and current information
- Explicit tool-calling loops
- Conversation persistence with `MemorySaver`
- Tests for SEO, X/Twitter, general handling, and follow-up memory

> This notebook prompts only for API keys.


## 1. Install Dependencies

Run this cell once in the notebook environment.


In [ ]:
# Section 1: Install the packages required by the final design.
%pip install -qU langgraph langchain langchain-openai deepagents langchain-tavily typing-extensions pydantic


## 2. Secure API-Key Setup

Keys are read from environment variables when available. If a key is missing, the notebook prompts for it securely. No other user input is requested.


In [ ]:
# Section 2: Read only the required API keys.
import getpass
import os


def require_api_key(env_name: str, prompt: str) -> str:
    key = os.getenv(env_name)
    if not key:
        key = getpass.getpass(prompt)
    if not key:
        raise ValueError(f"{env_name} is required.")
    os.environ[env_name] = key
    return key


require_api_key("OPENAI_API_KEY", "Enter your OpenAI API key: ")
require_api_key("TAVILY_API_KEY", "Enter your Tavily API key: ")


## 3. Imports and Model Configuration

GPT-5 is used for routing, writing, and deep research. The deep research agent gets its own model instance so its behavior and cost can be tuned independently later.


In [ ]:
# Section 3: Import the framework components and configure GPT-5.
from typing import Annotated, Literal

from deepagents import create_deep_agent
from langchain_core.messages import AIMessage, AnyMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from pydantic import BaseModel, Field
from typing_extensions import TypedDict


llm = ChatOpenAI(
    model="gpt-5",
    max_retries=3,
    timeout=120,
)

research_llm = ChatOpenAI(
    model="gpt-5",
    max_retries=3,
    timeout=180,
)


## 4. Shared Graph State

`messages` uses the `add_messages` reducer, so every node contributes to the same conversation history instead of replacing it.


In [ ]:
# Section 4: Define the typed state shared by every graph node.


class AgentState(TypedDict):
    user_input: str
    route: Literal["seo_blog_writer", "x_blog_writer", "general"]
    output: str
    messages: Annotated[list[AnyMessage], add_messages]


## 5. Advanced Tavily Search Tool

This is the lightweight search path. It is configured for advanced search and is appropriate for current SEO keywords, trends, supporting facts, and hashtags.


In [ ]:
# Section 5: Create the advanced Tavily search tool used by writers and researchers.
internet_search = TavilySearch(
    max_results=10,
    topic="general",
    search_depth="advanced",
    include_answer=True,
    include_raw_content=False,
)


@tool("internet_search_tool")
def internet_search_tool(query: str) -> str:
    """Search the current internet for keywords, trends, sources, and supporting facts."""
    result = internet_search.invoke({"query": query})
    return str(result)


## 6. Deep Research Sub-Agent

The deep researcher uses advanced Tavily search internally, synthesizes the results, and returns one research brief to the SEO writer. The prompt limits research to one or two focused search passes to control cost and latency.


In [ ]:
# Section 6: Create a focused Deep Research sub-agent.
deep_research_prompt = """
You are a senior content research specialist.

Produce a research brief for an SEO blog writer. Use the Tavily search tool for
one or two focused searches only. Prefer primary sources, official reports,
peer-reviewed research, and reputable publications. Do not fabricate facts.

Your final brief must include:
1. Topic scope and search intent
2. Five to seven key points with useful explanations
3. Important statistics or expert findings, with source URLs when available
4. Current or emerging angles
5. Content gaps and opportunities
6. Uncertainties or conflicting evidence

Return only the synthesized research brief.
"""


deep_research_agent = create_deep_agent(
    model=research_llm,
    tools=[internet_search_tool],
    system_prompt=deep_research_prompt,
)


@tool("research_tool")
def research_tool(query: str) -> str:
    """Run focused deep research and return a synthesized research brief."""
    result = deep_research_agent.invoke(
        {"messages": [{"role": "user", "content": query}]}
    )
    final_message = result["messages"][-1]
    return str(final_message.content)


## 7. Structured Router

A structured route schema prevents malformed labels from breaking conditional graph edges. The router also appends the current user message to persistent state.


In [ ]:
# Section 7: Define and configure the request router.


class RouteDecision(BaseModel):
    route: Literal["seo_blog_writer", "x_blog_writer", "general"] = Field(
        description="The specialist that should handle the request."
    )


router_llm = llm.with_structured_output(RouteDecision)

router_prompt = """
Classify the user request into exactly one route:

- seo_blog_writer: long-form blogs, articles, SEO content, or detailed content
- x_blog_writer: tweets, X posts, Twitter threads, hashtags, or short social content
- general: greetings, questions, follow-ups, or anything else
"""


def router_node(state: AgentState) -> dict:
    # Section 7a: Classify the request and persist the current user message.
    decision = router_llm.invoke(
        [
            SystemMessage(content=router_prompt),
            HumanMessage(content=state["user_input"]),
        ]
    )
    return {
        "route": decision.route,
        "messages": [HumanMessage(content=state["user_input"])],
    }


## 8. SEO Blog Writer

The SEO writer can call deep research and advanced internet search. It uses research first when substantive evidence is needed, then search for current SEO angles and keywords.


In [ ]:
# Section 8: Configure the SEO writer and its two tools.
seo_tools = [research_tool, internet_search_tool]
seo_llm = llm.bind_tools(seo_tools)

seo_prompt = """
You are an expert SEO blog writer.

Use research_tool for deep evidence and references. Use internet_search_tool
for current SEO keywords, trends, and supporting information. Use research_tool
at most once unless the topic clearly contains two independent research needs.
Use internet_search_tool only when it improves accuracy or SEO relevance.

Final response requirements:
- Write a complete long-form blog post.
- Include an engaging title, introduction, Markdown H2/H3 headings, and a conclusion.
- Use clear paragraphs, lists where useful, and natural keyword placement.
- Include a specific call to action.
- Do not mention internal tools or the workflow.
- Return only the finished blog post.
"""


def seo_blog_writer_node(state: AgentState) -> dict:
    # Section 8a: Continue the SEO agent with the complete message history.
    response = seo_llm.invoke(
        [SystemMessage(content=seo_prompt), *state["messages"]]
    )
    update = {"messages": [response]}
    if not response.tool_calls and response.content:
        update["output"] = response.content
    return update


## 9. X/Twitter Writer

The X writer receives only the advanced internet search tool and must keep its final response under 280 characters.


In [ ]:
# Section 9: Configure the X/Twitter writer with only the search tool.
x_tools = [internet_search_tool]
x_llm = llm.bind_tools(x_tools)

x_prompt = """
You are an expert X/Twitter content writer.

Use internet_search_tool when current trends, news, or hashtags would improve
the post. Avoid repeated searches for the same query.

Final response requirements:
- Keep the entire post under 280 characters.
- Use punchy, clear language.
- Add appropriate emojis.
- Include one to three relevant hashtags.
- Return only the final post, with no explanation or character count.
"""


def x_blog_writer_node(state: AgentState) -> dict:
    # Section 9a: Continue the X writer with the complete message history.
    response = x_llm.invoke(
        [SystemMessage(content=x_prompt), *state["messages"]]
    )
    update = {"messages": [response]}
    if not response.tool_calls and response.content:
        update["output"] = response.content
    return update


## 10. General Handler

The general handler uses the entire persisted message history. The router already stores the current user message, so this node only appends the AI response and avoids duplicating the user message.


In [ ]:
# Section 10: Define the context-aware general assistant.
general_prompt = """
You are a helpful general-purpose assistant.

Use the full conversation history to answer clearly and naturally. Handle
greetings, questions, and follow-up requests. If the user asks about a previous
request, inspect the persisted messages and answer from that context.
"""


def general_handler_node(state: AgentState) -> dict:
    # Section 10a: Answer with persisted context and append the AI message.
    response = llm.invoke(
        [SystemMessage(content=general_prompt), *state["messages"]]
    )
    return {
        "output": response.content,
        "messages": [response],
    }


## 11. Graph Routing and Tool Loops

Each content agent loops through its own `ToolNode` until the latest AI message contains no tool calls. The general handler terminates immediately.


In [ ]:
# Section 11: Build the LangGraph workflow.
seo_tool_node = ToolNode(seo_tools)
x_tool_node = ToolNode(x_tools)


def route_from_router(state: AgentState) -> str:
    # Section 11a: Send the request to the selected specialist.
    return state["route"]


def should_continue_seo(state: AgentState) -> str:
    # Section 11b: Continue SEO tool execution only when requested by the model.
    last_message = state["messages"][-1]
    return "seo_tools" if getattr(last_message, "tool_calls", None) else END


def should_continue_x(state: AgentState) -> str:
    # Section 11c: Continue X tool execution only when requested by the model.
    last_message = state["messages"][-1]
    return "x_tools" if getattr(last_message, "tool_calls", None) else END


builder = StateGraph(AgentState)

builder.add_node("router", router_node)
builder.add_node("seo_writer", seo_blog_writer_node)
builder.add_node("x_writer", x_blog_writer_node)
builder.add_node("general_handler", general_handler_node)
builder.add_node("seo_tools", seo_tool_node)
builder.add_node("x_tools", x_tool_node)

builder.add_edge(START, "router")

builder.add_conditional_edges(
    "router",
    route_from_router,
    {
        "seo_blog_writer": "seo_writer",
        "x_blog_writer": "x_writer",
        "general": "general_handler",
    },
)

builder.add_conditional_edges(
    "seo_writer",
    should_continue_seo,
    {"seo_tools": "seo_tools", END: END},
)
builder.add_edge("seo_tools", "seo_writer")

builder.add_conditional_edges(
    "x_writer",
    should_continue_x,
    {"x_tools": "x_tools", END: END},
)
builder.add_edge("x_tools", "x_writer")
builder.add_edge("general_handler", END)


## 12. Persistence and Compilation

Use the same `thread_id` for a conversation. `MemorySaver` is intentionally lightweight and persists while this Python process is alive.


In [ ]:
# Section 12: Compile the graph with a checkpointer.
memory = MemorySaver()
graph = builder.compile(checkpointer=memory)


# Optional visual inspection of the compiled workflow.
try:
    display(graph.get_graph().draw_mermaid())
except NameError:
    pass


## 13. Invocation Helper

This helper always passes a valid `thread_id`, preventing checkpointer errors and preserving follow-up context.


In [ ]:
# Section 13: Define a reusable invocation helper.
def run_request(user_input: str, thread_id: str = "demo-thread") -> dict:
    # Section 13a: Invoke the graph with the required persistence configuration.
    run_config = {"configurable": {"thread_id": thread_id}}
    return graph.invoke(
        {
            "user_input": user_input,
            "route": "general",
            "output": "",
            "messages": [],
        },
        config=run_config,
    )


## 14. Test the SEO Route

This invokes the full SEO writer and may perform deep research plus advanced search.


In [ ]:
# Section 14: Test the SEO blog route.
seo_result = run_request(
    "Write a detailed, research-backed blog post about AI in healthcare.",
    thread_id="test-seo",
)
print("Route:", seo_result["route"])
print(seo_result["output"])


## 15. Test the X/Twitter Route


In [ ]:
# Section 15: Test the X/Twitter route.
x_result = run_request(
    "Write a current X post about the future of artificial intelligence.",
    thread_id="test-x",
)
print("Route:", x_result["route"])
print("Character count:", len(x_result["output"]))
print(x_result["output"])
assert len(x_result["output"]) < 280


## 16. Test the General Route and Conversation Memory

The second request reuses the same thread and asks about the first request.


In [ ]:
# Section 16: Test general handling and persisted follow-up context.
general_result = run_request(
    "Hello. Please remember that I am building a content assistant.",
    thread_id="test-general",
)
print("Route:", general_result["route"])
print(general_result["output"])

follow_up_result = run_request(
    "What was my previous request in this conversation?",
    thread_id="test-general",
)
print("Follow-up route:", follow_up_result["route"])
print(follow_up_result["output"])


## 17. Validate All Routes


In [ ]:
# Section 17: Confirm that all three route types were exercised.
assert seo_result["route"] == "seo_blog_writer"
assert x_result["route"] == "x_blog_writer"
assert general_result["route"] == "general"
assert follow_up_result["route"] == "general"
print("✅ All routes and the persistence follow-up completed successfully.")


## 18. Optional Node-by-Node Trace

Use this helper when debugging tool calls or explaining the workflow. It is not run automatically because it makes another model invocation.


In [ ]:
# Section 18: Define an optional streaming trace helper.
def trace_request(user_input: str, thread_id: str = "trace-thread") -> None:
    # Section 18a: Stream updates so routing, tools, and final output are visible.
    run_config = {"configurable": {"thread_id": thread_id}}
    initial_state = {
        "user_input": user_input,
        "route": "general",
        "output": "",
        "messages": [],
    }
    for update in graph.stream(initial_state, config=run_config, stream_mode="updates"):
        print(update)
